# Graph-KV 单卡真实模型测试 Notebook

本 Notebook 专门用于 **单 GPU** 调试 Graph-KV。

## GPU 使用方式

只需要修改：

```python
GPU_ID = 2
```

例如：

- `GPU_ID = 0` → 物理 GPU 0
- `GPU_ID = 1` → 物理 GPU 1
- `GPU_ID = 2` → 物理 GPU 2
- `GPU_ID = 3` → 物理 GPU 3

本 Notebook **不使用 Bash 的 `CUDA_VISIBLE_DEVICES`**。

模型通过：

```python
device_map={"": GPU_ID}
```

整体加载到指定 GPU。

因此这是单卡模型加载，不会再自动分到 4 张 GPU。

> 注意：必须在加载模型时指定 `device_map={"": GPU_ID}`。不要再使用 `device_map="auto"`。


In [1]:
# ============================================================
# 1. 用户只需要修改这里
# ============================================================

GPU_ID = 2

MODEL_INDEX = 0
MODELS_ROOT = "/home/lyh/hcy_work/GraphKV/models"

# 是否运行真实 Llama Graph-KV 测试
RUN_REAL_MODEL_TEST = True

print("Requested physical GPU:", GPU_ID)


Requested physical GPU: 2


In [2]:
# ============================================================
# 2. 环境检查
# ============================================================

import sys
import os
import gc
import importlib
import traceback
from pathlib import Path

import torch

assert torch.cuda.is_available(), "CUDA is required."

gpu_count = torch.cuda.device_count()
assert 0 <= GPU_ID < gpu_count, (
    f"GPU_ID={GPU_ID} invalid. "
    f"Visible GPU count={gpu_count}"
)

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Visible GPU count:", gpu_count)

for i in range(gpu_count):
    free_b, total_b = torch.cuda.mem_get_info(i)
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)} | "
        f"free={free_b/2**30:.2f} GB / {total_b/2**30:.2f} GB"
    )

print()
print("Selected GPU:")
print(f"GPU {GPU_ID}: {torch.cuda.get_device_name(GPU_ID)}")


torch: 2.5.1+cu121
CUDA: 12.1
Visible GPU count: 4
GPU 0: NVIDIA GeForce RTX 4090 | free=21.67 GB / 23.64 GB
GPU 1: NVIDIA GeForce RTX 4090 | free=21.00 GB / 23.64 GB
GPU 2: NVIDIA GeForce RTX 4090 | free=23.26 GB / 23.64 GB
GPU 3: NVIDIA GeForce RTX 4090 | free=23.26 GB / 23.64 GB

Selected GPU:
GPU 2: NVIDIA GeForce RTX 4090


## 3. 清除旧模块

Jupyter 很容易保留之前导入的 `graph_kv_*` 模块，因此这里显式清除并重新导入。


In [3]:
MODULES = [
    "graph_kv_cache",
    "graph_kv_adapter",
    "pcw",
    "pcw_parallel",
]

for name in MODULES:
    sys.modules.pop(name, None)

importlib.invalidate_caches()

ROOT = Path.cwd()

for filename in [
    "graph_kv_cache.py",
    "graph_kv_adapter.py",
    "pcw.py",
    "pcw_parallel.py",
]:
    path = ROOT / filename
    print(
        f"{filename}: exists={path.exists()} "
        f"path={path}"
    )

import graph_kv_cache
import graph_kv_adapter
import pcw
import pcw_parallel

assert hasattr(graph_kv_adapter, "_tokenize_one")

from graph_kv_cache import GraphKVCache
from pcw import gapemp, gapemp_appr
from pcw_parallel import gapemp_graph

print()
print("Loaded modules:")
print("graph_kv_cache :", graph_kv_cache.__file__)
print("graph_kv_adapter:", graph_kv_adapter.__file__)
print("pcw            :", pcw.__file__)
print("pcw_parallel   :", pcw_parallel.__file__)


graph_kv_cache.py: exists=True path=/home/lyh/hcy_work/GraphKV/graph_kv_cache.py
graph_kv_adapter.py: exists=True path=/home/lyh/hcy_work/GraphKV/graph_kv_adapter.py
pcw.py: exists=True path=/home/lyh/hcy_work/GraphKV/pcw.py
pcw_parallel.py: exists=True path=/home/lyh/hcy_work/GraphKV/pcw_parallel.py


/home/lyh/miniconda3/envs/graphkv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loaded modules:
graph_kv_cache : /home/lyh/hcy_work/GraphKV/graph_kv_cache.py
graph_kv_adapter: /home/lyh/hcy_work/GraphKV/graph_kv_adapter.py
pcw            : /home/lyh/hcy_work/GraphKV/pcw.py
pcw_parallel   : /home/lyh/hcy_work/GraphKV/pcw_parallel.py


In [4]:
# ============================================================
# 4. 检查依赖
# ============================================================

import transformers
import flash_attn

print("transformers:", transformers.__version__)
print("flash_attn:", flash_attn.__version__)

assert transformers.__version__ == "4.50.0"


transformers: 4.50.0
flash_attn: 2.7.4.post1


## 5. 查找本地模型

默认目录：

`/home/lyh/hcy_work/GraphKV/models`


In [5]:
MODELS_ROOT = Path(MODELS_ROOT)

model_dirs = sorted([
    p for p in MODELS_ROOT.iterdir()
    if p.is_dir() and (p / "config.json").exists()
])

assert len(model_dirs) > 0, f"No models found under {MODELS_ROOT}"

for i, p in enumerate(model_dirs):
    print(i, p.name)

selected_model_dir = model_dirs[MODEL_INDEX]
print()
print("Selected model:")
print(selected_model_dir)


0 Tulu3-Block-FT
1 Tulu3-RAG
2 Tulu3-SFT

Selected model:
/home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT


## 6. 加载模型到指定单卡

关键设置：

```python
device_map={"": GPU_ID}
```

不要使用：

```python
device_map="auto"
```

整个模型将放在指定 GPU 上。


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(
    str(selected_model_dir),
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model on GPU", GPU_ID)
print("IMPORTANT: using device_map={\"\": GPU_ID}; no automatic multi-GPU sharding.")

model = AutoModelForCausalLM.from_pretrained(
    str(selected_model_dir),
    torch_dtype=torch.bfloat16,
    device_map={"": GPU_ID},
    attn_implementation="flash_attention_2",
    low_cpu_mem_usage=True,
)

model.eval()

print()
print("Model class:", type(model).__name__)
print("model.device:", model.device)
print("hf_device_map:", getattr(model, "hf_device_map", None))

# 严格确认模型没有被拆到多张卡
devices = set()

for p in model.parameters():
    devices.add(str(p.device))

print("Parameter devices:", sorted(devices))

assert devices == {f"cuda:{GPU_ID}"}, (
    f"Model is not fully on GPU {GPU_ID}: {devices}"
)

print(f"PASS: entire model is on cuda:{GPU_ID}")


Loading model on GPU 2
IMPORTANT: using device_map={"": GPU_ID}; no automatic multi-GPU sharding.


Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.35it/s]


Model class: LlamaForCausalLM
model.device: cuda:2
hf_device_map: {'': 2}
Parameter devices: ['cuda:2']
PASS: entire model is on cuda:2


## 7. 定义最小真实测试数据

先使用很小的 4 个 context，验证 Graph-KV 的真实 Llama forward。


In [7]:
prefix = (
    "You are a helpful question answering assistant. "
    "Use the provided evidence to answer the question."
)

middle = ""

query = "What is the capital city of France?"

contexts = [
    "France is a country in Europe.",
    "Paris is the capital city of France.",
    "Berlin is the capital city of Germany.",
    "The Eiffel Tower is located in Paris.",
]

print("query:", query)
print("num contexts:", len(contexts))
for i, text in enumerate(contexts):
    print(f"[{i}] {text}")


query: What is the capital city of France?
num contexts: 4
[0] France is a country in Europe.
[1] Paris is the capital city of France.
[2] Berlin is the capital city of Germany.
[3] The Eiffel Tower is located in Paris.


## 8. GraphKVCache 基础位置测试

In [8]:
# GraphKVCache 的位置布局依赖 source_span，
# 因此必须先注册一个 source node，再计算 target/query position。

from types import SimpleNamespace

tiny_cache = SimpleNamespace(
    key_cache=[
        torch.zeros(
            1, 1, 4, 8,
            dtype=torch.bfloat16,
        )
    ],
    value_cache=[
        torch.zeros(
            1, 1, 4, 8,
            dtype=torch.bfloat16,
        )
    ],
)

# 先用 source_position_start=0 测试公共 API。
# 真正的 prefix offset 会由 gapemp()/gapemp_graph() 根据 prefix 长度设置。
gkv_test = GraphKVCache(source_position_start=0)
gkv_test.add_source("dummy_source", tiny_cache)

print("source span:", gkv_test.source_span)

print(
    "target round 1:",
    gkv_test.target_position_ids(
        token_count=4,
        round_idx=1,
        device=torch.device("cpu"),
    ).tolist()
)

print(
    "query after round 1:",
    gkv_test.query_position_ids(
        token_count=4,
        device=torch.device("cpu"),
    ).tolist()
)


source span: 4
target round 1: [[4, 5, 6, 7]]
query after round 1: [[4, 5, 6, 7]]


### 位置测试预期

对于 4-token source，应该得到：

```text
source logical range: [0, 4)
target round-1:      [4, 8)
query:                [8, 12)
```

这只是 `GraphKVCache` 的无 prefix 基础测试；真实 `gapemp()` 会根据 prefix 长度设置 source 的 logical 起点。


In [9]:
import inspect

print("GraphKVCache.target_position_ids signature:")
print(inspect.signature(GraphKVCache.target_position_ids))

print("GraphKVCache.query_position_ids signature:")
print(inspect.signature(GraphKVCache.query_position_ids))


GraphKVCache.target_position_ids signature:
(self, token_count: 'int', *, round_idx: 'int' = 1, device: 'Optional[torch.device]' = None) -> 'torch.Tensor'
GraphKVCache.query_position_ids signature:
(self, token_count: 'int', *, device: 'Optional[torch.device]' = None) -> 'torch.Tensor'


## 9. Graph-KV 构建诊断

上一版 Notebook 中四个 Graph-KV 函数都正常返回，但结果是空字符串。
这很可能意味着**第一步预测就是 EOS**，而不是 forward 失败。

这里不再只显示最终字符串，而是：

1. 构建真实 `GraphKVCache`
2. 打印 `gkv.summary()`
3. 执行一次 query forward
4. 打印首个预测 token
5. 打印 top-10 logits
6. 再执行最多 16 token 的短 greedy decode

这样可以区分“Graph-KV 错误”与“测试 prompt 导致立即 EOS”。


In [10]:
from pcw import _build_rag_graph_cache

prefix_len = int(
    tokenizer(
        prefix,
        return_tensors="pt",
        add_special_tokens=False
    ).input_ids.shape[-1]
)

query_ids = tokenizer(
    query,
    return_tensors="pt",
    add_special_tokens=False,
).input_ids.to(model.device)

max_length = 512

gkv, query_ids, source_ids, target_ids = _build_rag_graph_cache(
    tokenizer,
    model,
    getattr(model.model, "rotary_emb", None),
    prefix,
    query,
    contexts,
    top_k=None,
    max_length=max_length,
)

print("GraphKV summary:")
print(gkv.summary())

print()
print("source_ids:", source_ids)
print("target_ids:", target_ids)
print("query_ids:", query_ids.tolist())


GraphKV summary:
{'source_position_start': 17, 'prefix_token_count': 17, 'source_span': 10, 'max_round': 1, 'query_position_start': 37, 'nodes': {'source::0': {'kind': 'source', 'token_count': 7, 'logical_range': (17, 24), 'round': 0, 'physical_seq_len': 7}, 'source::1': {'kind': 'source', 'token_count': 8, 'logical_range': (17, 25), 'round': 0, 'physical_seq_len': 8}, 'source::2': {'kind': 'source', 'token_count': 8, 'logical_range': (17, 25), 'round': 0, 'physical_seq_len': 8}, 'source::3': {'kind': 'source', 'token_count': 10, 'logical_range': (17, 27), 'round': 0, 'physical_seq_len': 10}, 'target::0': {'kind': 'target', 'token_count': 7, 'logical_range': (27, 34), 'round': 1, 'physical_seq_len': 7}, 'target::1': {'kind': 'target', 'token_count': 8, 'logical_range': (27, 35), 'round': 1, 'physical_seq_len': 8}, 'target::2': {'kind': 'target', 'token_count': 8, 'logical_range': (27, 35), 'round': 1, 'physical_seq_len': 8}, 'target::3': {'kind': 'target', 'token_count': 10, 'logical_r

In [11]:
# Prepare the physical KV cache and logical/physical query positions.
past_key_values, position_ids, cache_position = gkv.prepare_query(
    query_ids,
    source_ids=source_ids,
    target_ids=target_ids,
)

print("query position_ids:", position_ids.tolist())
print("query cache_position:", cache_position.tolist())
print("physical past length:", past_key_values.key_cache[0].shape[-2])


query position_ids: [[37, 38, 39, 40, 41, 42, 43, 44]]
query cache_position: [83, 84, 85, 86, 87, 88, 89, 90]
physical past length: 83


In [12]:
# One-step prediction diagnostic.
with torch.no_grad():
    outputs = model(
        query_ids,
        past_key_values=past_key_values,
        position_ids=position_ids,
        cache_position=cache_position,
        use_cache=True,
    )

logits = outputs.logits[:, -1, :]
top_values, top_indices = torch.topk(logits, k=10, dim=-1)

print("Top-10 next-token predictions:")
for rank, (idx, value) in enumerate(zip(top_indices[0], top_values[0]), 1):
    token_id = int(idx.item())
    token_text = tokenizer.decode([token_id], skip_special_tokens=False)
    is_eos = tokenizer.eos_token_id is not None and token_id == tokenizer.eos_token_id
    print(
        f"{rank:2d}. id={token_id:6d} "
        f"logit={float(value):9.4f} "
        f"eos={is_eos!s:5s} "
        f"text={token_text!r}"
    )

first_token = int(top_indices[0, 0].item())
print()
print("FIRST TOKEN:", first_token)
print("FIRST TOKEN TEXT:", tokenizer.decode([first_token], skip_special_tokens=False))
print("EOS TOKEN ID:", tokenizer.eos_token_id)


Top-10 next-token predictions:
 1. id=128001 logit=  15.2500 eos=True  text='<|end_of_text|>'
 2. id= 95509 logit=  13.8750 eos=False text='Berlin'
 3. id=   791 logit=  13.2500 eos=False text='The'
 4. id= 16533 logit=  13.2500 eos=False text='Answer'
 5. id= 60704 logit=  12.6875 eos=False text='Paris'
 6. id= 20437 logit=  11.9375 eos=False text=' Berlin'
 7. id= 22559 logit=  11.9375 eos=False text=' Answer'
 8. id=  3923 logit=  11.6875 eos=False text='What'
 9. id=  9399 logit=  11.6250 eos=False text='answer'
10. id=  1270 logit=  11.5625 eos=False text="'\n"

FIRST TOKEN: 128001
FIRST TOKEN TEXT: <|end_of_text|>
EOS TOKEN ID: 128001


In [13]:
# Short greedy decode without hiding EOS.
generated = query_ids.clone()
current_input = query_ids
past = past_key_values
pos = position_ids
cache_pos = cache_position

token_ids = []

with torch.no_grad():
    for step in range(16):
        out = model(
            current_input,
            past_key_values=past,
            position_ids=pos,
            cache_position=cache_pos,
            use_cache=True,
        )
        past = out.past_key_values

        next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        tid = int(next_token.item())
        token_ids.append(tid)

        text = tokenizer.decode(
            [tid],
            skip_special_tokens=False,
        )

        print(f"step={step:02d} token_id={tid} text={text!r}")

        if tokenizer.eos_token_id is not None and tid == int(tokenizer.eos_token_id):
            print("Generation stopped at EOS.")
            break

        generated = torch.cat([generated, next_token], dim=-1)
        current_input = next_token
        pos = pos[:, -1:] + 1
        cache_pos = cache_pos[-1:] + 1

print()
print("raw generated ids:", token_ids)
print(
    "decoded answer:",
    tokenizer.decode(
        token_ids,
        skip_special_tokens=False,
    )
)


step=00 token_id=128001 text='<|end_of_text|>'
Generation stopped at EOS.

raw generated ids: [128001]
decoded answer: <|end_of_text|>


## 10. 普通 Llama baseline

如果 Graph-KV 的首 token 是 EOS，需要先判断模型本身是否也会对同类最小 prompt 立即结束。

这里使用完全普通的 `model.generate()`，不引入 Graph-KV。


In [14]:
baseline_prompt = (
    prefix
    + "\n\nEvidence:\n"
    + "\n".join(f"- {x}" for x in contexts)
    + "\n\nQuestion: "
    + query
    + "\nAnswer:"
)

baseline_inputs = tokenizer(
    baseline_prompt,
    return_tensors="pt",
    add_special_tokens=False,
).to(model.device)

print("baseline prompt tokens:", baseline_inputs.input_ids.shape[-1])

with torch.no_grad():
    baseline_out = model.generate(
        **baseline_inputs,
        max_new_tokens=16,
        do_sample=False,
        use_cache=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

baseline_new = baseline_out[:, baseline_inputs.input_ids.shape[-1]:]

print("baseline generated ids:", baseline_new[0].tolist())
print(
    "baseline decoded:",
    tokenizer.decode(
        baseline_new[0].tolist(),
        skip_special_tokens=False,
    )
)


baseline prompt tokens: 68


/home/lyh/miniconda3/envs/graphkv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/lyh/miniconda3/envs/graphkv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


baseline generated ids: [578, 6864, 3363, 315, 9822, 374, 12366, 627, 128001]
baseline decoded:  The capital city of France is Paris.
<|end_of_text|>


## 11. 判断

### 情况 A：Graph-KV 首 token 是 EOS，但 baseline 也立即 EOS
这说明主要是**测试 prompt / 模型生成格式**问题，不能据此判断 Graph-KV 实现错误。

### 情况 B：Graph-KV 首 token 是 EOS，但 baseline 能正常生成
这才说明需要继续检查 Graph-KV KV 内容、attention mask 或 positional layout。

### 情况 C：Graph-KV 能生成，但回答明显异常
继续检查 source/target KV 的具体内容和 position audit。

原论文的 Graph-KV 核心是 source→target 的结构感知 KV 更新，以及 source/target 的共享 positional ranges。fileciteturn0file0L234-L275 fileciteturn0file0L279-L301


## 13. GPU 显存检查

In [15]:
torch.cuda.synchronize(GPU_ID)

free_b, total_b = torch.cuda.mem_get_info(GPU_ID)
allocated = torch.cuda.memory_allocated(GPU_ID)
reserved = torch.cuda.memory_reserved(GPU_ID)

print(f"GPU {GPU_ID}: {torch.cuda.get_device_name(GPU_ID)}")
print(f"free     : {free_b/2**30:.2f} GB")
print(f"allocated: {allocated/2**30:.2f} GB")
print(f"reserved : {reserved/2**30:.2f} GB")
print(f"total    : {total_b/2**30:.2f} GB")


GPU 2: NVIDIA GeForce RTX 4090
free     : 8.13 GB
allocated: 14.99 GB
reserved : 15.06 GB
total    : 23.64 GB


## 14. Position audit

如果 Graph-KV 前向成功，这里用于进一步检查：

- source 的 logical position
- target 的 logical position
- query 的 logical position
- physical KV cache position

这是后续验证论文 Graph-KV 位置编码设计的关键。


In [16]:
# 可选：在正式 debug 时打开
ENABLE_POSITION_TRACE = False

if ENABLE_POSITION_TRACE:
    original_forward = model.forward
    trace = []

    def traced_forward(*args, **kwargs):
        position_ids = kwargs.get("position_ids")
        cache_position = kwargs.get("cache_position")

        if position_ids is not None or cache_position is not None:
            trace.append({
                "position_ids": (
                    None
                    if position_ids is None
                    else position_ids.detach().cpu().tolist()
                ),
                "cache_position": (
                    None
                    if cache_position is None
                    else cache_position.detach().cpu().tolist()
                ),
                "input_len": (
                    args[0].shape[-1]
                    if args and torch.is_tensor(args[0])
                    else None
                ),
            })

        return original_forward(*args, **kwargs)

    model.forward = traced_forward

    try:
        _ = gapemp(
            tokenizer,
            model,
            getattr(model.model, "rotary_emb", None),
            prefix,
            middle,
            query,
            contexts,
            selected_model_dir.name,
            1.0,
            1.0,
            None,
        )
    finally:
        model.forward = original_forward

    for i, item in enumerate(trace):
        print(i, item)
else:
    print("ENABLE_POSITION_TRACE=False")
    print("Set it to True when you need detailed position debugging.")


ENABLE_POSITION_TRACE=False
Set it to True when you need detailed position debugging.


## 15. 释放模型

需要切换 GPU 或模型时，执行本 cell。


In [17]:
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

print(f"Released model from GPU {GPU_ID}")


Released model from GPU 2
